# W06B · A · 같은 잡음, 다른 클래스
2026-10-07 · 주교재5.1 · CPU · Python3.12/3.13 · 패키지0.1.10

목표: 클래스 조건과 정답 잡음을 구별하고, 실제 가중치 갱신을 확인한 뒤 같은 잡음에서 의류 종류를 바꿉니다. 이전 노트북 없이 첫 셀부터 실행하세요. Fashion-MNIST 약30 MB를 최초 다운로드하고 체크섬을 검사합니다. API 키와 개인 파일은 필요 없습니다.

[노트1](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/course/notion/w06b/w06b-conditioning.md) · [실습 기록지](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/course/notion/w06b/w06b-workbook.md) · [모든 함수 정의](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/W06B-API.md)


In [ ]:
import sys,subprocess,os,time
from pathlib import Path
from importlib.metadata import version,PackageNotFoundError
PUBLIC_REF='2026-fall-w06b'
try:
    ready=all(version(p)==v for p,v in [('luna-genai','0.1.10'),('transformers','5.15.1'),('diffusers','0.40.0'),('gradio','6.26.0')])
except PackageNotFoundError:
    ready=False
if not ready:
    if any(p in sys.modules for p in ('luna_genai','diffusers','gradio','transformers')):
        raise RuntimeError('이전 패키지가 이미 import되었습니다. 런타임을 재시작하고 첫 셀부터 실행하세요.')
    package=str(Path('src').resolve()) if Path('src/pyproject.toml').exists() else f'git+https://github.com/lunalab-ai/genAI.git@{PUBLIC_REF}#subdirectory=src'
    subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
torch.set_num_threads(2)
IN_COLAB='COLAB_RELEASE_TAG' in os.environ
print('Python',sys.version.split()[0],'luna-genai',version('luna-genai'),'torch',torch.__version__)
started=time.perf_counter()


## 1. 데이터와 라벨을 함께 준비
[`conditional.load_fashion`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional.py#L44)은 공식 데이터를 자동 준비하며 CPU 이미지 `(N,1,28,28)` [-1,1], int64 라벨 `(N,)`, 분할 인덱스를 반환합니다. 캐시 파일은 만들지만 모델을 학습하지 않습니다. [`FashionData`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional.py#L32)의 필드를 직접 확인하세요.


In [ ]:
from luna_genai.conditional import (load_fashion,TinyClassUNet,conditional_step,conditional_evaluation,sample_classes,load_conditional_checkpoint,CLASS_NAMES)
from luna_genai.diffusion import forward_noise,image_grid
data=load_fashion()
print('이번 다운로드:',data.downloads or '없음: 체크섬 검사 후 캐시 재사용')
print('train/validation/test:',data.train.shape,data.validation.shape,data.test.shape)
fig,axes=plt.subplots(2,5,figsize=(10,4))
for label,ax in enumerate(axes.flat):
    index=int((data.train_labels==label).nonzero()[0]);ax.imshow(data.train[index,0],cmap='gray',vmin=-1,vmax=1)
    ax.set_title(f'{label}: {CLASS_NAMES[label]}',fontsize=9);ax.axis('off')
plt.tight_layout();plt.show()


### A1 · 입력과 정답 예상하기
클래스7을 조건으로 넣을 때 모델 출력은 7이라는 숫자일까요, 잡음 배열일까요? 배치8에서 입력 세 개와 정답의 shape를 먼저 적으세요.


## 2. 한 배치를 손상시키고 실제로 예측
[`TinyClassUNet.__init__`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional.py#L88)은 새 가중치와 학습 가능한 클래스 임베딩을 만듭니다. [`TinyClassUNet.forward`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional.py#L93)는 noisy·time·label을 받아 잡음을 예측하며 모수를 갱신하지 않습니다. [`diffusion.forward_noise`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/diffusion.py#L49)는 정해진 cosine schedule로 배열을 섞습니다.


In [ ]:
fresh=TinyClassUNet(seed=20261007)
generator=torch.Generator().manual_seed(23)
x=data.train[:8];labels=data.train_labels[:8]
t=torch.randint(100,(len(x),),generator=generator)
epsilon=torch.randn(x.shape,generator=generator)
noisy=forward_noise(x,t,epsilon)
with torch.no_grad():prediction=fresh(noisy,t,labels)
display(pd.DataFrame([{'name':k,'shape':tuple(v.shape),'meaning':meaning} for k,v,meaning in [
    ('noisy',noisy,'손상 이미지'),('t',t,'시점'),('labels',labels,'클래스 조건'),('epsilon',epsilon,'정답 잡음'),('prediction',prediction,'예측 잡음')]]))
print('새 모델 MSE:',float(((prediction-epsilon)**2).mean()))


## 3. 32번의 실제 가중치 갱신
[`conditional_step`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional.py#L107)은 MSE → backward → optimizer.step을 실제 실행하고 loss·gradient_norm·가중치 변화량을 돌려줍니다. 바뀌는 것은 모델과 optimizer 상태입니다. 32회는 32epoch가 아닙니다. 이 짧은 실행의 목적은 학습 동작 확인입니다.


In [ ]:
optimizer=torch.optim.Adam(fresh.parameters(),lr=.0005)
history=[]
for step in range(32):
    ids=torch.randint(len(data.train),(32,),generator=generator)
    clean=data.train[ids];y=data.train_labels[ids]
    times=torch.randint(100,(len(clean),),generator=generator)
    target=torch.randn(clean.shape,generator=generator)
    history.append(conditional_step(fresh,optimizer,clean,y,times,target))
history=pd.DataFrame(history)
display(history.head());display(history.tail())
history.loss.plot(title='32 actual optimizer updates',xlabel='update index',ylabel='epsilon MSE');plt.show()


### A2 · backward와 step을 나눠 설명
`backward_parameter_change`와 `parameter_change`는 왜 다른가요? loss가 한 번 증가했다면 학습 코드가 반드시 틀린 것인지도 설명하세요.


## 4. 제공 체크포인트를 로드하고 미사용 데이터로 평가
[`load_conditional_checkpoint`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional.py#L169)는 SHA-256 검증 후 NPZ를 읽습니다. 누락되면 오류를 내며 무작위 모델로 대체하지 않습니다. 제공 모델은 공식 학습 집합59000장으로6000회 갱신했고 나머지1000장으로 점검했습니다. 아래 평가는 별도 공식 test 중128장을 사용합니다. 지금 수행한32회 학습과 구별하세요.

[`conditional_evaluation`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional.py#L126)은 고정 이미지·잡음에서 올바른 라벨과 `(label+1)%10`을 비교합니다. 잘못된 라벨의 MSE가 모든 시점에서 반드시 더 크다는 보장은 없습니다. 이 값은 클래스 정확도나 생성 품질 점수가 아닙니다.


In [ ]:
trained,record=load_conditional_checkpoint()
display({k:record[k] for k in ['train_size','validation_size','steps','batch_size','learning_rate','ema_decay','seconds','sha256']})
evaluation=conditional_evaluation(trained,data.test[:128],data.test_labels[:128])
display(evaluation)


### A3 · 조건 효과를 비교하는 코드 수정
다음에는 클래스0·7·9를 같은 초기 잡음으로 비교합니다. 먼저 어떤 모습이 달라질지 예상하세요. 실행 후 `chosen_classes`를 다른 세 종류로 바꾸되 seed는 유지해 비교하세요.


In [ ]:
chosen_classes=[0,7,9]
result=sample_classes(trained,chosen_classes,seed=42,steps=30,matched_noise=True)
display(image_grid(result['images'],title='Same initial noise, different labels',columns=3))
print('초기 잡음 동일:',torch.equal(result['initial'][0],result['initial'][1]))
print('서로 다른 클래스 결과 평균 절대 차이:',float((result['images'][0]-result['images'][1]).abs().mean()))


### A4 · 의미가 잘못된 코드를 진단
친구가 `loss = ((prediction-labels[:,None,None,None])**2).mean()`을 사용했습니다. 실행될 수 있어도 무엇이 잘못됐나요? 정답 배열을 쓰는 표현과 이유를 작성하세요.


## 5. 마지막 웹앱에서 확인
[`build_conditional_app`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional_app.py#L64)은 준비된 모델을 연결해 앱을 만들 뿐 실행·다운로드·학습하지 않습니다. [`class_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional_app.py#L17)는 모델로 실제 생성하고, [`recorded_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional_app.py#L29)는 저장된 SD 결과를 읽습니다. [`cfg_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w06b/src/luna_genai/conditional_app.py#L40)는 작은 벡터 수계산입니다. 세 작업은 다른 증거입니다.


In [ ]:
from luna_genai.conditional_app import build_conditional_app,class_view,recorded_view,cfg_view,APP_CSS
app=build_conditional_app(trained)
class_image,class_settings=class_view(7,42,model=trained)
saved_image,saved_settings=recorded_view('cfg-7.5')
vector_figure,vector_settings=cfg_view(4);plt.close(vector_figure)
repeat=sample_classes(trained,chosen_classes,seed=42,steps=30,matched_noise=True)


In [ ]:
assert not np.intersect1d(data.train_ids,data.validation_ids).size
assert data.train.dtype==torch.float32 and data.train_labels.dtype==torch.long
assert prediction.shape==epsilon.shape and np.isfinite(history.loss).all()
assert (history.gradient_norm>0).all() and (history.parameter_change>0).all()
assert record['steps']==6000 and (evaluation.mse<evaluation.zero_mse).all()
assert torch.equal(result['images'],repeat['images'])
assert class_image.shape==(56,56) and class_settings['class_id']==7
assert saved_image.size==(512,512) and '재생' in saved_settings['mode']
assert np.allclose(vector_settings['guided'],[1.4,.7],atol=1e-6)
assert len(app.config['dependencies'])==3
print('W06B A 검증 통과 · 계산 구간',round(time.perf_counter()-started,2),'초')


In [ ]:
if IN_COLAB:
    app.launch(share=True,inline=False,prevent_thread_lock=True,css=APP_CSS)
else:
    print('앱 실행: app.launch(css=APP_CSS). 공유 URL은 실행 중인 런타임 동안만 유지됩니다.')


공유 링크에서 클래스7과9를 같은 seed로 생성하고, SD 기록 비교 버튼과 CFG 계산 버튼도 눌러 보세요. 출력 설정과 입력이 일치하는지 확인합니다. 기록지에는 예상·설정·관찰·해석의 한계를 나누어 적습니다.
